In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
import random

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    # other params...
)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
import os

llmGoogle = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key =os.getenv("google_api_key")
)

from langchain_core.messages import HumanMessage, SystemMessage

#  Simple llm app

## Using Langugage models

In [2]:
llm.invoke("Hello!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-02-08T16:29:03.7487314Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9571729178, 'load_duration': 9218368522, 'prompt_eval_count': 12, 'prompt_eval_duration': 76278519, 'eval_count': 10, 'eval_duration': 267768636, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019c3e15-ae14-7970-b561-b12708069ee9-0', usage_metadata={'input_tokens': 12, 'output_tokens': 10, 'total_tokens': 22})

In [3]:
llm.invoke([{"role": "user","content":"Hello"}])

AIMessage(content='Hello! How are you today? Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-02-08T16:29:04.559355516Z', 'done': True, 'done_reason': 'stop', 'total_duration': 800014564, 'load_duration': 105578078, 'prompt_eval_count': 11, 'prompt_eval_duration': 62257347, 'eval_count': 23, 'eval_duration': 616422895, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019c3e15-d38e-7721-a51f-fdcb9a6f72cf-0', usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34})

In [7]:
llm.invoke([HumanMessage("Hello")])

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-02-01T15:00:52.232755566Z', 'done': True, 'done_reason': 'stop', 'total_duration': 418542171, 'load_duration': 112667937, 'prompt_eval_count': 11, 'prompt_eval_duration': 44573610, 'eval_count': 10, 'eval_duration': 254030001, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019c19b8-8fe5-7e31-a13f-9f49b3bc3831-0', usage_metadata={'input_tokens': 11, 'output_tokens': 10, 'total_tokens': 21})

## Streaming

In [9]:
for token in llm.stream([HumanMessage("Hello")]):
    print(token.content, end="|")

Hello|!| How| are| you| today|?| Is| there| something| I| can| help| you| with| or| would| you| like| to| chat|?|||

## Prompt Template

In [12]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [13]:
prompt = prompt_template.invoke({"language": "Spanish", "text": "Bye!"})

prompt

ChatPromptValue(messages=[SystemMessage(content='Translate the following from English into Spanish', additional_kwargs={}, response_metadata={}), HumanMessage(content='Bye!', additional_kwargs={}, response_metadata={})])

In [14]:
prompt.to_messages()

[SystemMessage(content='Translate the following from English into Spanish', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Bye!', additional_kwargs={}, response_metadata={})]

In [16]:
response = llm.invoke(prompt)
print(response.content)

¡Hasta luego!


# Semantic search

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Elephants are the largest land animals, known for their intelligence and strong social bonds.",
        metadata={"source": "wildlife-doc"},
        id="doc1",
    ),
    Document(
        page_content="Tigers are powerful predators, recognized by their distinctive striped fur.",
        metadata={"source": "wildlife-doc"},
        id="doc2",
    ),
    # Add more documents as needed
]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./langchain-exercises-main/docs/example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

107


In [24]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FO

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './langchain-exercises-main/docs/example_data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}


## Splitting

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

516

In [28]:
print(all_splits[0].page_content)

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE FISCAL YEAR ENDED MAY 31, 2023
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE TRANSITION PERIOD FROM                         TO                         .
Commission File No. 1-10635
NIKE, Inc.
(Exact name of Registrant as specified in its charter)
Oregon 93-0584541
(State or other jurisdiction of incorporation) (IRS Employer Identification No.)
One Bowerman Drive, Beaverton, Oregon 97005-6453
(Address of principal executive offices and zip code)
(503) 671-6453
(Registrant's telephone number, including area code)
SECURITIES REGISTERED PURSUANT TO SECTION 12(B) OF THE ACT:
Class B Common Stock NKE New York Stock Exchange
(Title of each class) (Trading symbol) (Name of each exchange on which registered)
